In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
train_df.shape, test_df.shape

((8693, 14), (4277, 13))

In [4]:
train_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [5]:
train_df.shape, test_df.shape

((8693, 14), (4277, 13))

In [6]:
def engineer(df: pd.DataFrame) -> pd.DataFrame: # Feature engineering
    out = df.copy() # Avoid modifying original df

    # --- Cabin split as three features to capture more info
    out[['CabinDeck','CabinNum','CabinSide']] = out['Cabin'].str.split('/', expand=True)
    out['CabinNum'] = pd.to_numeric(out['CabinNum'], errors='coerce')

    # --- Group features (based on PassengerId)
    out['Group'] = out['PassengerId'].astype(str).str.split('_').str[0] # Extract group ID by splitting at '_' and taking first part
    group_counts = out.groupby('Group')['PassengerId'].transform('count') # group size is count of passengers with same group ID
    # first groupby creates groups, second transform('count') counts members in each group and assigns to each member
    # then we assign this to a new column 'GroupSize'
    out['GroupSize'] = group_counts 
    out['IsAlone'] = (out['GroupSize'] == 1).astype(int) # 1 if alone, else 0

    # --- Spending features
    spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
    out['TotalSpending'] = out[spend_cols].sum(axis=1, skipna=True) # combine spendings, skipna=True to ignore NaNs in sum
    out['NoSpending'] = (out['TotalSpending'].fillna(0) == 0).astype(int) # 1 if no spending, else 0

    # --- Age bins to capture age groups
    out['AgeBin'] = pd.cut(out['Age'], bins=[-1, 12, 18, 30, 50, 100], labels=False)

    return out

train_fe = engineer(train_df)
test_fe  = engineer(test_df)

In [7]:
train_fe.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,Transported,CabinDeck,CabinNum,CabinSide,Group,GroupSize,IsAlone,TotalSpending,NoSpending,AgeBin
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,...,False,B,0.0,P,0001,1,1,0.0,1,3.0
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,...,True,F,0.0,S,0002,1,1,736.0,0,2.0
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,...,False,A,0.0,S,0003,2,0,10383.0,0,4.0
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,...,False,A,0.0,S,0003,2,0,5176.0,0,3.0
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,...,True,F,1.0,S,0004,1,1,1091.0,0,1.0


In [8]:
TARGET = 'Transported'
IDCOL  = 'PassengerId'

y = train_fe[TARGET].astype(int)
X = train_fe.drop(columns=[TARGET])

drop_cols = [IDCOL, 'Name', 'Cabin', 'Group']

In [9]:
# 4) Column typing
def get_feature_columns(df: pd.DataFrame): # Identify numeric and categorical columns
    use = df.drop(columns=drop_cols, errors='ignore') # Drop unused cols if present
    numeric_cols = use.select_dtypes(include=[np.number]).columns.tolist() # Numeric cols
    categorical_cols = use.select_dtypes(exclude=[np.number]).columns.tolist() # Categorical cols
    return numeric_cols, categorical_cols

num_cols, cat_cols = get_feature_columns(X) # Get feature columns

spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','TotalSpending'] 

In [10]:
# Preprocessing

num_zero_cols = [c for c in spend_cols if c in num_cols] # first find which spend cols are numeric then use those for zero imputation 
num_median_cols = [c for c in num_cols if c not in num_zero_cols] # rest of numeric cols use median imputation

numeric_transformers = [] # list of (name, transformer, columns) tuples for numeric columns, list will be later combined with categorical transformers
if num_zero_cols: # only add if there are such columns
    numeric_transformers.append(( # append a tuple with name, transformer pipeline, and columns
        'num_zero',
        Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value=0))]),
        num_zero_cols
    ))
if num_median_cols: 
    numeric_transformers.append(( # append a tuple with name, transformer pipeline, and columns
        'num_median',
        Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]),
        num_median_cols
    ))

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # fill missing with most frequent category
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # one-hot encode, ignore unknown categories in test set
])

preprocess = ColumnTransformer( # combine all transformers
    transformers = numeric_transformers + [ # transformers will be list of numeric transformers plus categorical transformer
        ('cat', categorical_transformer, cat_cols) # categorical transformer applied to categorical columns
    ], # total there are 3 transformers: num_zero, num_median, cat
    remainder='drop' # drop any columns not specified in transformers
)


In [11]:
# 6) Train/Validation split
# ---------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X.drop(columns=drop_cols, errors='ignore'),
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [12]:
# 7) Model
# xgb = XGBClassifier( 
#     random_state=42,
#     eval_metric='logloss',
#     tree_method='hist',
#     n_estimators=300,
#     max_depth=3,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8
# )

# trying catboost
cb = CatBoostClassifier(
    random_state=42,
    eval_metric='Logloss',
    iterations=2000,
    depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bylevel=0.8,
    early_stopping_rounds=100,
    verbose=0
)


pipe = Pipeline(steps=[ # combine preprocessing and model into one pipeline
    ('preprocess', preprocess),
    ('model', cb)
])

In [13]:
# from sklearn.model_selection import cross_val_score, StratifiedKFold
# from sklearn.metrics import accuracy_score

# # --- Cross-validation on the training set ---
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

# print("CV scores:", scores)
# print("Mean CV accuracy:", scores.mean())

In [14]:
# 8) Train and validate
pipe.fit(X_train, y_train)
y_val_pred = pipe.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)

print(f"✅ Validation Accuracy: {val_acc*100:.2f}%")

✅ Validation Accuracy: 82.12%


In [15]:
# 9) Retrain on full data & predict test
pipe.fit(X.drop(columns=drop_cols, errors='ignore'), y) # retrain on full data as we have no more validation set

X_test = test_fe.drop(columns=drop_cols, errors='ignore') # drop unused cols if present
test_pred = pipe.predict(X_test).astype(bool)

submission = pd.DataFrame({
    'PassengerId': test_fe[IDCOL],
    'Transported': test_pred
})
submission.to_csv('submission_fixed.csv', index=False)
print(f"Predictions saved to 'submission_fixed.csv'")
print(f"Number of predictions: {len(submission)}")
submission.head()

Predictions saved to 'submission_fixed.csv'
Number of predictions: 4277


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [16]:
# # hyperparameter tuning xgboost using randomized search with cross-validation
# from sklearn.model_selection import GridSearchCV
# # --- IGNORE ---
# param_grid = {
#     'model__n_estimators': [100, 200, 300, 400],
#     'model__max_depth': [3, 5, 7],
#     'model__learning_rate': [0.01, 0.05, 0.1],
#     'model__subsample': [0.6, 0.8, 1.0],
#     'model__colsample_bytree': [0.6, 0.8, 1.0]
# }
# grid_search = GridSearchCV(pipe, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)
# grid_search.fit(X_train, y_train)
# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")


In [17]:
# # hyperparameter tuning for catboost using randomized search with cross-validation
# from sklearn.model_selection import RandomizedSearchCV
# # --- IGNORE ---
# param_dist = {
#     'model__iterations': [100, 200, 300, 400],
#     'model__depth': [3, 5, 7],
#     'model__learning_rate': [0.01, 0.05, 0.1],
#     'model__subsample': [0.6, 0.8, 1.0],
#     'model__colsample_bylevel': [0.6, 0.8, 1.0]
#     # 'model__l2_leaf_reg': [1, 3, 5, 7, 9]
# # --- IGNORE ---
# }
# random_search = RandomizedSearchCV(pipe, param_dist, n_iter=20, cv=3, scoring='accuracy', n_jobs=-1, verbose=2, random_state=42)
# random_search.fit(X_train, y_train)
# print(f"Best parameters: {random_search.best_params_}")
# print(f"Best cross-validation accuracy: {random_search.best_score_:.4f}")


In [18]:
# # 8. Train model with proper validation
# X_train, X_val, y_train, y_val = train_test_split(
#     X_train_processed, y_train_full, test_size=0.2, random_state=42
# )

# # Train the model
# # rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
# # rf_model.fit(X_train, y_train)

In [19]:
# # Hyperparameter tuning can be done here if needed
# from sklearn.model_selection import GridSearchCV
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [None, 10, 20],
#     'min_samples_split': [2, 5, 10]
# }
# grid_search = GridSearchCV(rf_model, param_grid, cv=3, scoring='accuracy')
# grid_search.fit(X_train, y_train)


In [20]:
# # apply the best model
# rf_model = grid_search.best_estimator_


In [21]:
# # Validate the model
# y_val_pred = rf_model.predict(X_val)
# val_accuracy = accuracy_score(y_val, y_val_pred)
# print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

In [22]:
# # accuracy for xgboost
# accuracy_xgb = accuracy_score(y_val, test_predictions)
# print(f"XGBoost Validation Accuracy: {accuracy_xgb * 100:.2f}%")


In [23]:
# # Create submission file
# submission = pd.DataFrame({
#     'PassengerId': test_df['PassengerId'],
#     'Transported': test_predictions_gb
# })

# submission.to_csv('submission_fixed.csv', index=False)
# print(f"Predictions saved to 'submission_fixed.csv'")
# print(f"Number of predictions: {len(submission)}")
# submission.head()